In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.pyplot import imshow
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2

In [ ]:
image_height = 64
image_width = 64
image_size = (image_height,image_width)
batch_size = 32

train_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
     r"D:\friday_20_dataset\train",
     target_size=image_size,
     batch_size=batch_size,
     class_mode='binary',
     shuffle=True
)
val_generator = train_datagen.flow_from_directory(
    r"D:\friday_20_dataset\val",
    target_size=image_size,
    batch_size=batch_size,
    class_mode='binary',  
    shuffle=False
)      

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    r"D:\friday_20_dataset\test",         
    target_size=image_size,
    batch_size=batch_size,
    class_mode='binary',
    # classes=['benign', 'ddos'],
    shuffle=False
)    

In [ ]:
base_model = MobileNetV2(
    input_shape=(64, 64, 3),
    weights='imagenet',
    include_top=False
)
base_model.trainable = False

In [ ]:
from tensorflow.keras.layers import Conv2D, GlobalAveragePooling2D, Flatten, Dense, Activation, BatchNormalization, Dropout
t_model = keras.models.Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(8, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])
t_model.summary()

In [ ]:
from keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

t_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy',metrics=['accuracy', 
             keras.metrics.Precision(),
             keras.metrics.Recall()]
               )



callback = EarlyStopping(
    monitor='val_loss',           
    patience=5,                   
    restore_best_weights=True,    
    verbose=1                     
)

In [ ]:
history = t_model.fit(train_generator, validation_data = val_generator, epochs=30, callbacks=[callback],verbose=1)

In [ ]:
metrics = t_model.evaluate(val_generator)
print("loss:",metrics[0] )
print("Accuracy:", metrics[1])

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('')
plt.ylabel('Accuracy')
plt.xlabel('Epochs')
plt.legend(['Train Accuracy', 'Validation Accuracy'])
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend(['Train Loss', 'Validation Loss'])
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

test_generator.reset()
final_results = t_model.evaluate(test_generator, verbose=1)

print(f"Test Accuracy: {final_results[1]:.4f}")
print(f"Test Loss: {final_results[0]:.4f}")
print(f"Test Precision: {final_results[2]:.4f}")
print(f"Test Recall: {final_results[3]:.4f}")

y_pred = t_model.predict(test_generator)
y_pred_classes = (y_pred > 0.5).astype(int).flatten()
y_true = test_generator.classes

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred_classes))
print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes, target_names=['Benign', 'DDoS']))


cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Benign', 'DDoS'],
            yticklabels=['Benign', 'DDoS'],
            cbar=True, 
            square=True,
            )

plt.title('Confusion Matrix', fontsize=16, weight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()